# Split cleaned data into relevant tables

We want to split the cleaned csv into seperate tables:

orders - containing all our order data: transaction ID, user ID, product code, timestamp, quantity, price.

products - containing all our product data: codes / descriptions

customers - userID and their country

outliers_price - outliers in product price, flagged for manual review 

outliers_quantity - outliers in product order quantity, flagged for manual review 

In [3]:
%pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 32.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [statsmodels] [statsmodels]

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from statsmodels.stats.stattools import medcouple
from pathlib import Path

In [9]:
PROJECT_DIR = Path("/Users/hamzashah/data-platform-engineering-snowflake")

input_path = PROJECT_DIR / "datasets" / "cleaned" / "cleaned_transactions.csv"
output_dir = PROJECT_DIR / "datasets" / "cleaned"

# Load the cleaned dataset
df = pd.read_csv(input_path)

As we saw in the 1st notebook - 01-ingest-and-clean.ipynb :

Cost per item has a very large positive skew, whereas numbers of items purchased (quantity) does not, this shows us that perhaps we should not use styatistical methods such as IQR or Z score to calculate outliers, as the large positive skew would make them poor fits for this type of data.

Both measures (Zscore and IQR) assume a normally distributed set of data, the large positive skew on quantity would push the averages higher and hide large orders (masking).

A better fit would be to use a Medcouple; which is a statistical measure which accounts for skewness.

This is also available in python using __statsmodels.stats.stattools.medcouple__


## Creating outliers tables (outliers_price, and outliers_quantity)

In [23]:
# only look at columns which are numeric
numeric_cols = df.select_dtypes(include="number").columns.difference(
    ["userid", "transactionid", "transactiontime", "itemcode"])

def count_adjusted_boxplot_outliers(series):
    values = series.dropna().to_numpy(dtype=float)

    q1 = np.quantile(values, 0.25)
    q3 = np.quantile(values, 0.75)
    iqr = q3-q1

    mc = medcouple(values, use_fast = True)

    # Adjust fences according to skewness
    if mc >= 0:
        lower = q1 - 1.5 * np.exp(-4 * mc) * iqr
        upper = q3 + 1.5 * np.exp(3 * mc) * iqr
    else:
        lower = q1 - 1.5 * np.exp(-3 * mc) * iqr
        upper = q3 + 1.5 * np.exp(4 * mc) * iqr

    return ((values < lower) | (values > upper)).sum()

# check for statistical outliers using medcouples
outliers_df = pd.DataFrame({
    "Column": numeric_cols,
    "Outliers": [
        count_adjusted_boxplot_outliers(df[col])
        for col in numeric_cols
    ],
    # display skew
    "Skew": [
        df[col].skew()
    for col in numeric_cols
    ],
    "Medcouple": [
        medcouple(df[col].dropna().to_numpy(dtype=float), use_fast=True)
        for col in numeric_cols
    ]
})

display(outliers_df)


#create mask to filter dataframe
def adjusted_boxplot_outlier_mask(series):
    values = series.to_numpy(dtype=float)
    valid = ~np.isnan(values)
    clean_values = values[valid]

    q1 = np.quantile(clean_values, 0.25)
    q3 = np.quantile(clean_values, 0.75)
    iqr = q3 - q1

    mc = medcouple(clean_values, use_fast=True)

    if mc >= 0:
        lower = q1 - 1.5 * np.exp(-4 * mc) * iqr
        upper = q3 + 1.5 * np.exp(3 * mc) * iqr
    else:
        lower = q1 - 1.5 * np.exp(-3 * mc) * iqr
        upper = q3 + 1.5 * np.exp(4 * mc) * iqr

    mask = pd.Series(False, index=series.index)
    mask.loc[valid] = (
        (clean_values < lower) | (clean_values > upper)
    )

    return mask, lower, upper


# save the outliers into a csv
output_dir.mkdir(parents=True, exist_ok=True)

# Identify outliers independently
price_mask, price_lower, price_upper = (
    adjusted_boxplot_outlier_mask(df["costperitem"])
)

quantity_mask, quantity_lower, quantity_upper = (
    adjusted_boxplot_outlier_mask(df["numberofitemspurchased"])
)

# Keep complete transaction rows
outliers_price = df.loc[price_mask].copy()
outliers_quantity = df.loc[quantity_mask].copy()

# Export
outliers_price.to_csv(
    output_dir / "outliers_price.csv", index=False
)
outliers_quantity.to_csv(
    output_dir / "outliers_quantity.csv", index=False
)



,Column,Outliers,Skew,Medcouple
0,costperitem,7763,625.514333,0.33333333333333304
1,numberofitemspurchased,11581,400.328330,0.3333333333333333


## Creating Orders table

In [26]:
order_columns = [
    "transactionid",
    "userid",
    "itemcode",
    "transactiontime",
    "transaction_date",
    "transaction_clock_time",
    "numberofitemspurchased",
    "costperitem",
]

orders = df[order_columns].copy()

orders.to_csv(
    output_dir / "orders.csv",
    index = False
)

## Creating Products table

In [25]:
product_columns = [
    "itemcode",
    "itemdescription"
]

orders = (df[product_columns].drop_duplicates(subset=["itemcode"]).reset_index(drop=True))

orders.to_csv(
    output_dir / "products.csv",
    index = False
)

## Creating Customers table

In [27]:
customers_columns = [
    "userid",
    "country"
]

orders = (df[customers_columns].drop_duplicates(subset=["userid"]).reset_index(drop=True))

orders.to_csv(
    output_dir / "customers.csv",
    index = False
)